# What is a time series?

Before you start: upload `daily_sales.csv` to Colab.


The data is made up: three years of daily sales for a fictional shop.

## Load the data

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv("daily_sales.csv")
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,sales
0,2022-01-01,86
1,2022-01-02,179
2,2022-01-03,146
3,2022-01-04,163
4,2022-01-05,133


In [3]:
print(df.shape)
print(df["date"].min(), "to", df["date"].max())

(1096, 2)
2022-01-01 00:00:00 to 2024-12-31 00:00:00


## Look at the whole series

In [4]:
fig = px.line(df, x="date", y="sales", title="Daily sales, 3 years")
fig.show()

## Piece 1: the trend
A 30-day average smooths out the ups and downs, so we see the direction.

In [5]:
df["trend"] = df["sales"].rolling(30).mean()

fig = px.line(df, x="date", y=["sales", "trend"], title="Sales and 30-day average")
fig.show()

## Piece 2: the weekly pattern
Zoom in on the last 8 weeks.

In [6]:
last_weeks = df.tail(56)

fig = px.line(last_weeks, x="date", y="sales", markers=True, title="Last 8 weeks")
fig.show()

In [8]:
df["weekday"] = df["date"].dt.day_name()

by_day = df.groupby("weekday")["sales"].mean()
by_day = by_day.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

fig = px.bar(by_day, title="Average sales by day of the week")
fig.show()

## Piece 3: the yearly pattern

In [9]:
df["month"] = df["date"].dt.month

by_month = df.groupby("month")["sales"].mean()

fig = px.line(by_month, markers=True, title="Average sales by month")
fig.show()

## Piece 4: the odd days (holidays)

In [10]:
lowest = df.sort_values("sales").head(8)
lowest[["date", "sales"]]

,date,sales
358,2022-12-25,84
723,2023-12-25,85
327,2022-11-24,86
0,2022-01-01,86
730,2024-01-01,108
691,2023-11-23,112
365,2023-01-01,112
9,2022-01-10,126


## Splitting a time series
With time series, the test set is the **end** of the data, never a random sample.

In [11]:
train = df.iloc[:-60]
test = df.iloc[-60:]

print("Train:", train["date"].min().date(), "to", train["date"].max().date())
print("Test: ", test["date"].min().date(), "to", test["date"].max().date())

Train: 2022-01-01 to 2024-11-01
Test:  2024-11-02 to 2024-12-31


What goes wrong with a random split? Pick 3 random days as the 'test'. Their neighbors (the days right before and after) are in the training data, so the model has basically already seen the answer.

In [12]:
random_test = df.sample(3, random_state=1)

for day in random_test["date"]:
    before = df[df["date"] == day - pd.Timedelta(days=1)]["sales"].values[0]
    actual = df[df["date"] == day]["sales"].values[0]
    after = df[df["date"] == day + pd.Timedelta(days=1)]["sales"].values[0]
    print(day.date(), "| day before:", before, "| this day:", actual, "| day after:", after)

2024-09-24 | day before: 393 | this day: 400 | day after: 404
2023-03-12 | day before: 283 | this day: 250 | day after: 197
2024-10-27 | day before: 453 | this day: 439 | day after: 372
